# STINTSY Major Course Output

### Group 3
### Members:

* Rohann Gabriel D. Dizon
* Rainier A. Dulatre

### Dataset Description

* The dataset comprises competitive gameplay statistics sourced from **Tetrio**, a modern, fast-paced multiplayer Tetris clone. It specifically tracks the performance and match data of the top 500 players in the Tetra League. The data is scraped from 61,935 match replays of the top 500 Tetra League players between January 22, 2024, and January 30, 2024.

* Source: Kaggle (https://www.kaggle.com/datasets/n3koasakura/tetr-io-top-players-replays/data)

### Structure of the Data
* ##### **Unit of Analysis**: Each individual row represents a single subframe piece placement (one block dropped) 
* ##### **Number of Observations/Rows**: 7,716,524 piece placements.
* ##### **Format**: Tabular data with 21 variables/columns.

* ### Description of Variables

| **Features/Variables** | **Description** |
|------------------------|-----------------|
| game_id | The unique ID for this game/replay (players in the same match get different IDs). |
| subframe | The subframe of this placement. There 60 frames per second and 10 subframes per frame, or 600 subframes per second, starting with subframe 0. |
| won | 1 if this placement is part of a replay that won its match. Otherwise, 0. |
| playfield | The 10x40 playfield right before the placement occured. The cells of the playfield are ordered left to right, bottom to top. The playfield is truncated once all the remaining cells are empty. An empty string is not a missing value, but instead represents a playfield where all cells are empty. Each cell is represented with the letter of its corresponding tetrimino, except for empty and garbage cells, which are represented with "N" and "G" respectively. |
| x | The x-coordinate of the center of the placed piece according to the "true rotations" used in the Super Rotation System. 0 indicates the leftmost column, 9 indicates the rightmost column. |
| y | The y-coordinate of the center of the placed piece according to the "true rotations" used in the Super Rotation System. 0 indicates the bottommost row, 39 indicates the topmost row. |
| r | The rotation of the placed piece according to the Super Rotation System. N(0) - North; E(R) - East; S(2) - South; W(L) - West  |
| placed | The type of piece placed. |
| hold | The type of piece currently being held. "N" means no piece is being held. |
| next | The types of the next 14 peices in the queue (only the first 5 are visible to the player). |
| cleared | The number of rows cleared with this placement. |
| garbage_cleared | The number of rows containing garbage cleared with this placement. |
| attack | The rows of garbage sent with this placement, before garbage blocking. |
| t_spin | The type of T-spin performed. N - None; M - Mini; F - Full |
| btb | The length of the back-to-back chain right before this placement. |
| combo | The length of the combo chain right before this placement. |
| immediate_garbage | The rows of incoming garbage in queue ready to be received (the actual amount that will be received is capped at 8). |
| incoming_garbage | The total rows of incoming garbage in queue. |
| rating | The player's Tetra League rating (25000 is the maximum rating). |
| glicko | The player's Tetra League Glicko-2 rating. |
| glicko_rd | The player's Tetra League Glicko-2 rating deviation. |

### Variables Used in this Study

This study used **14** of the 21 variables, which are `game_id`, `subframe`, `won`, `cleared`, `garbage_cleared`, `attack`, `t_spin`, `btb`, `combo`, `immediate_garbage`, `incoming_garbage`, `rating`, `glicko`, and `glicko_rd`.The reason for choosing these variables is its relevance to the group's desired prediction task for the models to be tackled later in the notebook. However, note that **these are not the final variables used in model training/validation/testing** as it proceeds through data cleaning and feature engineering.

### Importing Libraries

The following libraries are used throughout the project and are necessary to perform data preprocessing, analysis, visualizations, and model training, validation, and testing:

In [40]:
# Standard Library & System
import sys
import os
import itertools

# Data Manipulation & Visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    confusion_matrix, 
    ConfusionMatrixDisplay
)

# Pytorch
import torch
import torch.optim as optim
import torch.nn as nn

# Custom Local Modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.neural_network import NeuralNetwork
from src.data_loader import DataLoader

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Raw Contents of the Dataset

In [41]:
df = pd.read_csv("../data/data.csv")

In [42]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7716524 entries, 0 to 7716523
Data columns (total 21 columns):
 #   Column             Dtype  
---  ------             -----  
 0   game_id            int64  
 1   subframe           int64  
 2   won                int64  
 3   playfield          str    
 4   x                  int64  
 5   y                  int64  
 6   r                  str    
 7   placed             str    
 8   hold               str    
 9   next               str    
 10  cleared            int64  
 11  garbage_cleared    int64  
 12  attack             int64  
 13  t_spin             str    
 14  btb                int64  
 15  combo              int64  
 16  immediate_garbage  int64  
 17  incoming_garbage   int64  
 18  rating             float64
 19  glicko             float64
 20  glicko_rd          float64
dtypes: float64(3), int64(12), str(6)
memory usage: 1.2 GB


In [43]:
df.shape

(7716524, 21)

In [44]:
df.head()

,game_id,subframe,won,playfield,x,y,r,placed,hold,next,...,garbage_cleared,attack,t_spin,btb,combo,immediate_garbage,incoming_garbage,rating,glicko,glicko_rd
0,1,67,1,NaN,4,0,N,I,N,JZSOTLSLIOJZTJ,...,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624
1,1,170,1,NNNIIII,4,1,N,Z,J,SOTLSLIOJZTJTZ,...,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624
2,1,339,1,NNNIIIINNNNNNNZZNNNNNNNZZ,6,1,E,S,J,OTLSLIOJZTJTZS,...,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624
3,1,472,1,NNNIIIISNNNNNNZZSSNNNNNZZNS,8,0,N,O,J,TLSLIOJZTJTZSO,...,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624
4,1,602,1,NNNIIIISOONNNNZZSSOONNNZZNS,0,1,E,J,T,LSLIOJZTJTZSOI,...,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624


The code results above gives us insight about what the data looks like before being preprocessed. This also lets us verify that we are indeed dealing with 7,716,524 rows and 21 columns.

## Data Preprocessing

Prior to data exploration and analysis, data cleaning and preprocessing will be carried out to resolve any discrepancies in the dataset and avoid inaccurate results.

The following was checked and applied with the necessary cleaning procedures:

* Removing Unnecessary Variables
* Multiple Representations
* Missing Data
* Negative Values
* Incorrect Datatypes
* Default Values and Inconsistent Formatting
* Duplicate Data
* Data Inconsistencies
* Outliers
* Feature Engineering

### Removing Unnecessary Variables

As previously mentioned, this study only focused on *14* specific variables that are related to the prediction task, so other columns that are not needed are disregarded, specifically variables related to the spatial playfield.

In [45]:
unneeded_cols = ["playfield", "x", "y", "r", "next", "hold", "placed"]
present_unneeded = [col for col in unneeded_cols if col in df.columns]
df = df.drop(columns=present_unneeded)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7716524 entries, 0 to 7716523
Data columns (total 14 columns):
 #   Column             Dtype  
---  ------             -----  
 0   game_id            int64  
 1   subframe           int64  
 2   won                int64  
 3   cleared            int64  
 4   garbage_cleared    int64  
 5   attack             int64  
 6   t_spin             str    
 7   btb                int64  
 8   combo              int64  
 9   immediate_garbage  int64  
 10  incoming_garbage   int64  
 11  rating             float64
 12  glicko             float64
 13  glicko_rd          float64
dtypes: float64(3), int64(10), str(1)
memory usage: 824.2 MB


### Checking For Multiple Representations

This is done to check for multiple represenations in the categorical variables that will be used

In [46]:
df["won"].value_counts()

won
1    4274700
0    3441824
Name: count, dtype: int64

In [47]:
df["t_spin"].value_counts()

t_spin
N    7150022
F     527604
M      38898
Name: count, dtype: int64

The number of unique labels and values within `won` and `t_spin` are correctly represented and conform with the unique labels defined by the dataset's dictionary

### Checking For Missing Data

This is done to identify missing data or null values in each variable

In [48]:
df.isnull().any()

game_id              False
subframe             False
won                  False
cleared              False
garbage_cleared      False
attack               False
t_spin               False
btb                  False
combo                False
immediate_garbage    False
incoming_garbage     False
rating               False
glicko               False
glicko_rd            False
dtype: bool

No further processing is required regarding missing data as there are no presence of null values in any of the variables

### Checking for Negative Values

This is done to check for negative values, especially for the variables with a continuous numerical value. Given the context of the dataset and variables, there should ideally be no negative values.

In [49]:
numeric_cols = df.select_dtypes(include=["number"])
negative_counts = (numeric_cols < 0).sum()
print(negative_counts)

game_id              0
subframe             0
won                  0
cleared              0
garbage_cleared      0
attack               0
btb                  0
combo                0
immediate_garbage    0
incoming_garbage     0
rating               0
glicko               0
glicko_rd            0
dtype: int64


Results show that no negative values were found within the variables

### Checking for Incorrect Datatypes

This is done to check if the variables are currently assigned appropriate variables

In [50]:
df.dtypes

game_id                int64
subframe               int64
won                    int64
cleared                int64
garbage_cleared        int64
attack                 int64
t_spin                   str
btb                    int64
combo                  int64
immediate_garbage      int64
incoming_garbage       int64
rating               float64
glicko               float64
glicko_rd            float64
dtype: object

It shows that all variables are already assigned the appropriate variables

However, the current variables with numerical datatypes (int and float) could use a smaller-range for memory optimization purposes, especially useful during model training

In [51]:
int_cols = [
    "won",
    "game_id",
    "subframe",
    "cleared",
    "garbage_cleared",
    "attack",
    "btb",
    "combo",
    "immediate_garbage",
    "incoming_garbage",
]
float_cols = ["rating", "glicko", "glicko_rd"]
for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], downcast="integer", errors="coerce")
for col in float_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], downcast="float", errors="coerce")

df.dtypes

game_id                int32
subframe               int32
won                     int8
cleared                 int8
garbage_cleared         int8
attack                  int8
t_spin                   str
btb                     int8
combo                   int8
immediate_garbage       int8
incoming_garbage        int8
rating               float32
glicko               float32
glicko_rd            float32
dtype: object

### Checking for Default Values and Inconsistent Formatting

This is done to check for distinct values in every variable and identify whether it contains default values or inconsistent formatting among the values

In [52]:
for col in df:
    print(f"{col}: {df[col].unique()}")
    print("")

game_id: [     1      2      3 ... 124030 124031 124032]

subframe: [    67    170    339 ... 102556 103224 103661]

won: [1 0]

cleared: [0 2 1 4 3]

garbage_cleared: [0]

attack: [ 0  4  1  5  3  6  2  7  8 10  9 11 12 14 15 16 17 13 19 18 22 20]

t_spin: <StringArray>
['N', 'F', 'M']
Length: 3, dtype: str

btb: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54]

combo: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21]

immediate_garbage: [ 0  7  8  4  2  1  6 10 13  5  3  9 11 15 12 14 20 16 19 23 22 18 28 17
 21 26 25 24 31 27 32 36 30 40 35 41 33 29 34 38 37 69 78 49 42 50 43 51
 39 46 59 58 77 61 52 56]

incoming_garbage: [ 0  7 15  9  2  1  4  6  5  3 13 10 11  8 16 12 14 18 20 19 23 29 17 28
 22 24 21 26 25 27 31 32 34 36 30 40 35 41 33 38 37 69 78 49 42 50 43 51
 39 46 66 58 77 74 52 56 59 44]

rating: [24748.521 24820.71  24788.744 24830.146 

Based from the results, there are no default values nor inconsistent formatting present in any of the variables

However, it can be seen that the only value for `garbage_cleared` is **0**, which makes it essentially useless and unnecessary to keep moving forward.

In [53]:
df = df.drop(columns=['garbage_cleared'])

### Checking for Duplicate Data

This is done to check for any rows that have the exact same values

In [54]:
df.duplicated().any()

np.True_

Results show that there are rows with the exact same values, which should not be possible especially if rows in one game contain the same subframes as it is physically impossible to have two or more pieces placed within the same subframe

In [55]:
df = df.drop_duplicates(subset=["game_id", "subframe"])
df.shape

(7716512, 13)

This shows that 12 rows have been removed from the intial amount of rows after duplicate rows with the same `game_id` and `subframe`

### Checking for Inconsistent Games

This is to check for inconsistencies within the games. Variables like a player's starting `rating`, `glicko`, `glicko_rd`, and whether they `won` the match should be identical for every single piece placed within the same game

In [56]:
if {"game_id", "won", "rating", "glicko", "glicko_rd"}.issubset(df.columns):
    gchecks = df.groupby("game_id")[["won", "rating", "glicko", "glicko_rd"]].nunique()
    inconsistent_games = gchecks[(gchecks > 1).any(axis=1)].index.tolist()
    print(f"Inconsistent games found: {len(inconsistent_games)}")
    df = df[~df["game_id"].isin(inconsistent_games)]

Inconsistent games found: 0


Based on the results, no inconsistencies where found within the games

### Checking for Outliers

This is done to check for outliers within the numerical variables the will be used. This was done through statistical and frequency-based outlier detection, specifically doing a percentile and frequency check. This process was chosen over using visualizations since we are dealing with 7.7 million rows for now

In [57]:
num_cols = [
    "cleared",
    "attack",
    "btb",
    "combo",
    "immediate_garbage",
    "incoming_garbage",
]
for col in num_cols:
    if col in df.columns:
        high = df[col].quantile(0.999)
        low = df[col].quantile(0.001)
        print(f"{col}: min={df[col].min()}, max={
              df[col].max()}, 0.1%={low}, 99.9%={high}")

for col in num_cols:
    if col in df.columns:
        top_values = df[col].value_counts().nlargest(5).to_dict()

        max_val = df[col].max()

        print(f"--- {col} ---")
        print(f"Max Value: {max_val}")
        print(f"Top 5 most common values: {top_values}\n")

cleared: min=0, max=4, 0.1%=0.0, 99.9%=4.0
attack: min=0, max=22, 0.1%=0.0, 99.9%=7.0
btb: min=0, max=54, 0.1%=0.0, 99.9%=19.0
combo: min=0, max=21, 0.1%=0.0, 99.9%=7.0
immediate_garbage: min=0, max=78, 0.1%=0.0, 99.9%=15.0
incoming_garbage: min=0, max=78, 0.1%=0.0, 99.9%=17.0
--- cleared ---
Max Value: 4
Top 5 most common values: {0: 5587283, 1: 958295, 2: 690330, 4: 381767, 3: 98837}

--- attack ---
Max Value: 22
Top 5 most common values: {0: 6033910, 1: 590235, 4: 439143, 2: 207244, 5: 197437}

--- btb ---
Max Value: 54
Top 5 most common values: {0: 3580652, 1: 1467989, 2: 890306, 3: 553606, 4: 388236}

--- combo ---
Max Value: 21
Top 5 most common values: {0: 5596511, 1: 1315812, 2: 445764, 3: 197292, 4: 88032}

--- immediate_garbage ---
Max Value: 78
Top 5 most common values: {0: 6690409, 1: 243668, 5: 150719, 4: 149941, 2: 148039}

--- incoming_garbage ---
Max Value: 78
Top 5 most common values: {0: 5866748, 1: 380035, 5: 290027, 4: 277661, 2: 265844}



The results show that the data for `cleared` is perfect and expected since a player can clear 0, 1, 2, 3, or 4 lines (a Tetris) with a single piece.

Furthemore, extreme but valid gameplay spikes can be observed for `combo`. A 21-combo usually indicates a perfectly executed center 4-wide or side 4-wide setup, which is possible.

Moreover, spikes can be observed for `attack`, `btb`, `immediate_garbage`, and `incoming_garbage` as they similarly have a relatively low value for the 99.9% quantile but have a huge maximum value. Again, these are valid extremes and are possible in a game. In Tetrio, attack scales massively with B2B and combo multipliers. A B2B Tetris at the end of a 10+ combo can absolutely spike damage into the 20s in a single subframe, a 54 B2B chain means a player continuously maintained T-spins and Tetrises for almost the entire game, and 78 lines of incoming garbage means a player had a massive red bar of garbage waiting to enter their board and end the game.

**We decided to keep these extreme values as these are essentially highlights or game-winning moves of players. Considering that this data is from the Top 500 players, the extreme values are markers of specific and elite playstyles.**

### Feature Engineering

This is done to transform the data into more meaningful features by aggregating the data by their `game_id` to work on a game/match level as it adheres to our prediction goal that requires analysis on an entire match instead of individual piece placements

In [58]:
agg_dict = {
    "subframe": ["max", "count"],
    "cleared": "sum",
    "attack": "sum",
    "t_spin": lambda x: x.isin(["M", "F"]).sum(),
    "btb": ["mean", "max"],
    "combo": ["mean", "max"],
    "immediate_garbage": ["mean", "max"],
    "incoming_garbage": ["mean", "max"],
    "won": "first",
    "rating": "first",
    "glicko": "first",
    "glicko_rd": "first",
}

game_level = df.groupby("game_id").agg(agg_dict)

game_level.columns = ["_".join([c for c in col if c]) for col in game_level.columns]
game_level = game_level.reset_index()

rename_map = {
    "t_spin_<lambda>": "t_spin_count",
    "won_first": "won",
    "rating_first": "rating",
    "glicko_first": "glicko",
    "glicko_rd_first": "glicko_rd",
}
game_level = game_level.rename(columns=rename_map)

game_level["duration_sec"] = game_level["subframe_max"] / 600
game_level["pps"] = game_level["subframe_count"] / game_level["duration_sec"]
game_level["attack_per_piece"] = game_level["attack_sum"] / game_level["subframe_count"]
game_level["apm"] = (game_level["attack_sum"] / game_level["duration_sec"]) * 60
game_level["tspin_rate"] = game_level["t_spin_count"] / game_level["subframe_count"]

print("Num rows in game_level:", len(game_level))

Num rows in game_level: 76692


From 7,716,512 rows, we are now dealing with 76,692 rows, each of which representing one game/match. Along with this, most of columns have been transformed and is now utilizing mean and maximum values (`btb_mean`, `btb_max`, `combo_mean`, `combo_max`, `immediate_garbage_mean`, `immediate_garbage_max`, `incoming_garbage_mean`, `incoming_garbage_max`) and rates (`pps` or piece per second, `attack_per_piece`, `apm` or attack per minute, and	`tspin_rate`), while columns like `won`, `rating`, `glicko`, and `glicko_rd` have been retained. Additionally, a `duration_sec` was also made and represents the duration of the game, in seconds

With that, we decided to drop the games/matches that lasted for less than 10 seconds to rule out instant disconnections and forfeits

In [59]:
game_level = game_level[game_level["duration_sec"] >= 10.0].copy()
game_level.shape

(70246, 23)

Lastly, as the current data still contains intermediate columns from the process of aggregation, it has no more use and needs to be dropped. The game_id identifier must also be dropped as it should not be used during model training

In [60]:
cols_to_drop = [
    "game_id",
    "subframe_max",
    "subframe_count",
    "cleared_sum",   
    "attack_sum",
    "t_spin_count"
]

final_df = game_level.drop(columns=cols_to_drop)
final_df.head()

,btb_mean,btb_max,combo_mean,combo_max,immediate_garbage_mean,immediate_garbage_max,incoming_garbage_mean,incoming_garbage_max,won,rating,glicko,glicko_rd,duration_sec,pps,attack_per_piece,apm,tspin_rate
0,1.933673,8,0.520408,7,0.244898,8,0.576531,15,1,24748.521484,2701.887695,62.212624,70.600000,2.776204,0.821429,136.827195,0.076531
1,0.413953,4,0.590698,6,0.683721,13,1.432558,13,0,24820.710938,2791.866943,63.886936,71.310000,3.015005,0.586047,106.015987,0.055814
2,0.611842,3,0.605263,8,0.250000,6,0.605263,6,1,24820.710938,2791.866943,63.886936,49.275000,3.084729,0.671053,124.200913,0.065789
3,0.786765,4,0.588235,6,0.551471,9,1.227941,10,0,24748.521484,2701.887695,62.212624,49.283333,2.759554,0.625000,103.483260,0.036765
4,1.277487,7,0.539267,5,0.366492,11,0.691099,11,1,24748.521484,2701.887695,62.212624,69.525000,2.747213,0.670157,110.463862,0.057592


In [61]:
# Optionally export the processed data
# final_df.to_csv("../data/data_processed.csv", index=False)